In [ ]:
# ===========================
# Grokking δ-Law: mod-add & mod-mul (Colab-ready)
# ===========================
# - Trains a tiny Transformer on modular addition and multiplication
# - Uses small train split to induce late generalization ("grokking")
# - Logs train loss / val accuracy time-series
# - Computes δ = (D_KY - 1) * (τ - 2) on loss fluctuation series
# - Reports δ before/after grokking (val acc > 0.9)
# ===========================

import math, time, random, sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---- Repro ----
def set_seed(s=123):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

set_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ===========================
# Data generation (modular arithmetic)
# ===========================
def build_mod_dataset(N=97, op="add", train_frac=0.2, seed=123):
    rng = np.random.RandomState(seed)
    xs, ys = [], []
    for a in range(N):
        for b in range(N):
            if op == "add":
                y = (a + b) % N
            elif op == "mul":
                y = (a * b) % N
            else:
                raise ValueError("op must be 'add' or 'mul'")
            xs.append([a, b]); ys.append(y)
    xs, ys = np.array(xs, dtype=np.int64), np.array(ys, dtype=np.int64)
    # shuffle
    idx = rng.permutation(len(xs))
    xs, ys = xs[idx], ys[idx]
    # split
    n_train = int(len(xs) * train_frac)
    x_train, y_train = xs[:n_train], ys[:n_train]
    x_val,   y_val   = xs[n_train:], ys[n_train:]
    return x_train, y_train, x_val, y_val, N

# ===========================
# Tiny Transformer for sequence length 2
# ===========================
class TinyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, nlayers=2, d_ff=256, dropout=0.1):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, d_model)
        self.pos = nn.Embedding(2, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=d_ff, dropout=dropout, batch_first=True)
        self.enc = nn.TransformerEncoder(encoder_layer, num_layers=nlayers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        # x: [B,2]
        B, L = x.shape
        h = self.tok(x) + self.pos(torch.arange(L, device=x.device).unsqueeze(0).expand(B, L))
        h = self.enc(h)
        h = self.norm(h[:, 0, :])  # pool on first token
        logits = self.head(h)
        return logits

# ===========================
# Training loop with logging
# ===========================
@torch.no_grad()
def eval_acc(model, x, y, batch=2048):
    model.eval()
    acc = 0; n = 0
    for i in range(0, len(x), batch):
        xb = torch.from_numpy(x[i:i+batch]).to(device)
        yb = torch.from_numpy(y[i:i+batch]).to(device)
        logits = model(xb)
        pred = logits.argmax(dim=-1)
        acc += (pred == yb).sum().item()
        n += len(xb)
    return acc / max(n, 1)

def run_grokking(op="add", N=97, train_frac=0.2, max_steps=20000, batch_size=512,
                 d_model=128, nhead=4, nlayers=2, d_ff=256, wd=1e-3, lr=3e-3, dropout=0.1, seed=123):
    print(f"\n=== GROKKING RUN: op={op}, N={N}, train_frac={train_frac} ===")
    xtr, ytr, xva, yva, vocab = build_mod_dataset(N=N, op=op, train_frac=train_frac, seed=seed)
    model = TinyTransformer(vocab, d_model=d_model, nhead=nhead, nlayers=nlayers, d_ff=d_ff, dropout=dropout).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    criterion = nn.CrossEntropyLoss()

    # Pre-pack train tensors
    xtr_t = torch.from_numpy(xtr).to(device)
    ytr_t = torch.from_numpy(ytr).to(device)

    # Logs
    train_loss_series = []
    val_acc_series = []
    iters_to_eval = 100  # evaluate every 100 steps to speed up
    best_val = 0.0
    grok_step = None

    model.train()
    steps = 0
    rng = np.random.RandomState(seed)
    t0 = time.time()
    while steps < max_steps:
        # mini-batch sample
        idx = rng.randint(0, len(xtr_t), size=(batch_size,))
        xb = xtr_t[idx]; yb = ytr_t[idx]
        logits = model(xb)
        loss = criterion(logits, yb)

        opt.zero_grad()
        loss.backward()
        opt.step()

        train_loss_series.append(loss.item())
        steps += 1

        if steps % iters_to_eval == 0:
            vacc = eval_acc(model, xva, yva)
            val_acc_series.append((steps, vacc))
            if grok_step is None and vacc >= 0.90:
                grok_step = steps
            if vacc > best_val: best_val = vacc

    t1 = time.time()
    print(f"Finished {steps} steps in {t1-t0:.1f}s | Best val acc={best_val:.3f} | grok_step={grok_step}")
    return {
        "model": model,
        "xtr": xtr, "ytr": ytr, "xva": xva, "yva": yva,
        "train_loss_series": np.array(train_loss_series, dtype=np.float32),
        "val_acc_series": np.array(val_acc_series, dtype=np.float32),
        "grok_step": int(grok_step) if grok_step is not None else None,
        "N": N, "op": op
    }

# ===========================
# δ-Law metrics on a 1D series (pure NumPy)
# ===========================
def _embed(ts, m=3, delay=3):
    ts = np.asarray(ts, float).ravel()
    N = len(ts) - (m-1)*delay
    if N <= 10:
        return None
    X = np.empty((N, m), float)
    for i in range(m):
        X[:, i] = ts[i*delay:i*delay+N]
    return X

def lyapunov_rosenstein_np(ts, m=3, delay=3, evolve=60, exclude=30):
    X = _embed(ts, m=m, delay=delay)
    if X is None or len(X) < 200:
        return np.nan
    N = len(X)
    # brute nearest neighbor with temporal exclusion (O(N^2), but N small after subsample)
    nnei = []
    for i in range(N):
        di = np.linalg.norm(X[i] - X, axis=1)
        di[max(0, i-exclude):min(N, i+exclude+1)] = np.inf
        j = np.argmin(di)
        if np.isfinite(di[j]): nnei.append((i, j))
    if len(nnei) < 50:
        return np.nan
    L = min(evolve, N-1)
    div = []
    for h in range(1, L):
        vals = []
        for i, j in nnei:
            if i+h < N and j+h < N:
                d0 = np.linalg.norm(X[i] - X[j]) + 1e-12
                d1 = np.linalg.norm(X[i+h] - X[j+h]) + 1e-12
                vals.append(np.log(d1/d0))
        if len(vals) > 30:
            div.append(np.mean(vals))
    if len(div) < 10:
        return np.nan
    Xh = np.arange(len(div))
    Yh = np.array(div)
    # linear fit slope
    slope = np.polyfit(Xh, Yh, 1)[0]
    return float(max(slope, 0.0))

def D_KY_from_l1_proxy(l1):
    if not np.isfinite(l1) or l1 <= 0:
        return 1.0
    lam2 = 0.0
    lam3 = -1.0
    s12 = l1 + lam2
    return 2.0 + s12 / abs(lam3)

def tau_avalanche(series, pct=90, min_bursts=20):
    x = np.asarray(series, float).ravel()
    thr = np.percentile(x, pct)
    runs = []
    c = 0
    for v in x:
        if v > thr: c += 1
        else:
            if c > 0: runs.append(c)
            c = 0
    if c > 0: runs.append(c)
    if len(runs) < min_bursts:
        return np.nan
    runs = np.asarray(runs, float)
    runs.sort()
    ccdf = 1.0 - np.arange(1, len(runs)+1) / (len(runs)+1)
    X = np.log(runs + 1e-12)
    Y = np.log(ccdf + 1e-12)
    slope = np.polyfit(X, Y, 1)[0]  # slope ≈ -(τ-1)
    tau = 1 - slope
    return float(tau)

def delta_metric(DKY, tau):
    if not (np.isfinite(DKY) and np.isfinite(tau)):
        return np.nan
    return (DKY - 1.0) * (tau - 2.0)

def compute_delta_from_loss(loss_series, subsample=1, diff=True):
    s = np.asarray(loss_series, float)
    if diff:
        s = np.diff(s)
    # detrend
    s = s - np.mean(s)
    # downsample to keep NN O(N^2) manageable
    if subsample > 1:
        s = s[::subsample]
    # make positive observable for bursts
    s_abs = np.abs(s)
    l1 = lyapunov_rosenstein_np(s_abs, m=3, delay=3, evolve=80, exclude=50)
    DKY = D_KY_from_l1_proxy(l1)
    tau = tau_avalanche(s_abs, pct=90, min_bursts=20)
    delt = delta_metric(DKY, tau)
    return l1, DKY, tau, delt

def report_delta_around_grokking(loss_series, grok_step, window=3000, subsample=5):
    n = len(loss_series)
    if grok_step is None:
        # no grokking detected; evaluate last two windows
        a0, a1 = max(0, n//4 - window//2), min(n, n//4 + window//2)
        b0, b1 = max(0, 3*n//4 - window//2), min(n, 3*n//4 + window//2)
        l1a, D1, t1, d1 = compute_delta_from_loss(loss_series[a0:a1], subsample=subsample)
        l1b, D2, t2, d2 = compute_delta_from_loss(loss_series[b0:b1], subsample=subsample)
        return ("no-threshold", (l1a, D1, t1, d1), (l1b, D2, t2, d2))
    pre0, pre1 = max(0, grok_step - window), grok_step
    post0, post1 = grok_step, min(n, grok_step + window)
    l1a, D1, t1, d1 = compute_delta_from_loss(loss_series[pre0:pre1], subsample=subsample)
    l1b, D2, t2, d2 = compute_delta_from_loss(loss_series[post0:post1], subsample=subsample)
    return ("threshold", (l1a, D1, t1, d1), (l1b, D2, t2, d2))

def pretty_metrics(tag, tup):
    l1, D, t, d = tup
    return f"{tag}:  λ1={l1:.5f}  D_KY={D:.5f}  τ={t:.5f}  δ={d:.5f}"

# ===========================
# Run BOTH experiments
# ===========================
cfgs = [
    {"op": "add", "N": 97, "train_frac": 0.2, "max_steps": 15000, "batch_size": 512, "wd": 1e-3, "lr": 3e-3, "seed": 123},
    {"op": "mul", "N": 97, "train_frac": 0.2, "max_steps": 25000, "batch_size": 512, "wd": 2e-3, "lr": 3e-3, "seed": 321},
]

results = []
for cfg in cfgs:
    out = run_grokking(**cfg)
    results.append(out)

    # δ before vs after grokking threshold
    kind, pre, post = report_delta_around_grokking(out["train_loss_series"], out["grok_step"], window=4000, subsample=5)
    print("\n---- δ-Law around transition ----")
    if kind == "threshold":
        print(f"Grokking at step {out['grok_step']}")
    else:
        print("No 90% threshold, comparing early vs late windows")
    print(pretty_metrics("pre", pre))
    print(pretty_metrics("post", post))

print("\n=== DONE ===")

Device: cpu

=== GROKKING RUN: op=add, N=97, train_frac=0.2 ===
Finished 15000 steps in 1047.5s | Best val acc=0.002 | grok_step=None

---- δ-Law around transition ----
No 90% threshold, comparing early vs late windows
pre:  λ1=0.00000  D_KY=1.00000  τ=2.78700  δ=0.00000
post:  λ1=0.00000  D_KY=1.00000  τ=3.19556  δ=0.00000

=== GROKKING RUN: op=mul, N=97, train_frac=0.2 ===
Finished 25000 steps in 1888.8s | Best val acc=0.030 | grok_step=None

---- δ-Law around transition ----
No 90% threshold, comparing early vs late windows
pre:  λ1=0.00000  D_KY=1.00000  τ=3.06086  δ=0.00000
post:  λ1=0.00166  D_KY=2.00166  τ=2.97438  δ=0.97600

=== DONE ===
